# z005b - Revalidar 4 trials más de z005 (para completar 10)

Se corrió el 07/10 dentro del notebook del submit 12; se movió acá para que los notebooks de submit
solo generen archivos. Revalida con 5 semillas los 4 trials siguientes de z005 (train 202104, valid 202106).

Resultado: entraron al top 10 los trials 31 (392,8), 20 (392,7), 28 (392,2) y 39 (391,8).
La lista final quedó en `z005_top10_final.csv` (la usa `submits/12_ens100_z005.ipynb`).

In [1]:
# --- 1. Librerías y parámetros ---
import os
import time
import numpy as np
import pandas as pd
import lightgbm as lgb

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
SEMILLAS_REVAL = [501419, 623353, 786349, 834569, 946331]
SEMILLAS = SEMILLAS_REVAL + [111119, 222227, 333331, 444443, 555557]   # tus 5 + 5 nuevas
N_TRIALS_EXTRA = 4     # trials nuevos a revalidar
TOP_TRIALS = 10
MESES_FINAL = [202103, 202104, 202105, 202106]
MES_PRED = 202108
# Archivos a generar: (número de submit, N). El de 12.000 va último.
ARCHIVOS = [(12, 11_500), (13, 12_000)]
CARPETA = "probs_noche"
os.makedirs(CARPETA, exist_ok=True)

params_fijos = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
}

trials = pd.read_csv("../0_exploracion/z005_optuna_trials.csv", sep=";", decimal=",", encoding="utf-8-sig")
ranking = pd.read_csv("../0_exploracion/z005_top10_5semillas.csv", sep=";", decimal=",",
                      encoding="utf-8-sig", index_col="trial")

In [2]:
# --- 2. Cargar datos, target y lags (igual que z003) ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria"] = (df["clase_ternaria"] == "BAJA+2").astype(int)

no_usar = ["foto_mes", "clase_ternaria", "clase_binaria"]
originales = [c for c in df.columns if c not in no_usar]
cols = [c for c in originales if c != "numero_de_cliente"]

df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo
variables = [c for c in df.columns if c not in no_usar]
print("Variables:", len(variables))

Variables: 761


In [3]:
# --- 3. Revalidar trials extra con 5 semillas (train 202104, valid 202106) ---
train = df[df["foto_mes"] == 202104]
valid = df[df["foto_mes"] == 202106]
y_valid = valid["clase_binaria"].values
ganancia_cliente = np.where(y_valid == 1, GANANCIA_ACIERTO, -COSTO_ESTIMULO)
datos_train = lgb.Dataset(train[variables], label=train["clase_binaria"], free_raw_data=False)

# Siguientes trials en el ranking de 1 semilla que todavía no se revalidaron
extra = trials[~trials["number"].isin(ranking.index)].head(N_TRIALS_EXTRA)
for _, t in extra.iterrows():
    p = {**params_fijos, "num_leaves": int(t["params_num_leaves"]),
         "min_sum_hessian_in_leaf": float(t["params_min_sum_hessian_in_leaf"])}
    g = []
    for semilla in SEMILLAS_REVAL:
        modelo = lgb.train({**p, "seed": semilla}, datos_train, num_boost_round=int(t["user_attrs_best_iter"]))
        acum = np.cumsum(ganancia_cliente[np.argsort(modelo.predict(valid[variables]))[::-1]])
        g.append(acum.max() / 1e6)
    ranking.loc[int(t["number"]), ["ganancia_max_M_mean", "ganancia_max_M_std", "params_num_leaves",
                                   "params_min_sum_hessian_in_leaf", "user_attrs_best_iter", "optuna_1semilla_M"]] = \
        [round(np.mean(g), 1), round(np.std(g, ddof=1), 1), t["params_num_leaves"],
         t["params_min_sum_hessian_in_leaf"], t["user_attrs_best_iter"], t["value"]]
    print("Trial", int(t["number"]), "promedio 5 semillas:", round(np.mean(g), 1))

del datos_train, train, valid
ranking = ranking.sort_values("ganancia_max_M_mean", ascending=False)
elegidos = ranking.head(TOP_TRIALS)
elegidos[["ganancia_max_M_mean", "ganancia_max_M_std", "params_num_leaves",
          "params_min_sum_hessian_in_leaf", "user_attrs_best_iter"]]

Trial 20 promedio 5 semillas: 392.7


Trial 39 promedio 5 semillas: 391.8


Trial 31 promedio 5 semillas: 392.8


Trial 28 promedio 5 semillas: 392.2


,ganancia_max_M_mean,ganancia_max_M_std,params_num_leaves,params_min_sum_hessian_in_leaf,user_attrs_best_iter
trial,,,,,
34,399.6,2.3,137.0,9.755574,538.0
36,395.5,4.4,61.0,7.745374,727.0
27,394.6,3.8,241.0,5.632060,578.0
29,394.5,2.0,193.0,5.968488,567.0
37,393.6,3.2,240.0,0.003039,888.0
31,392.8,2.3,203.0,1.444970,532.0
20,392.7,2.9,159.0,0.021543,843.0
9,392.5,2.5,65.0,1.250592,684.0
28,392.2,4.9,237.0,0.095672,826.0
